# 02 · Coupures

Anna Preto — AstroDeep Project

L'entonnoir des huit coupures dans l'ordre de la recette, une galerie de rejets par coupure sur pixels avec la raison dessinée, les grandeurs continues autour de chaque seuil, et le contrôle des biais en magnitude, taille et ellipticité.

Lit les produits de la chaîne locale (`res config/local_demo.yaml all`). Les sorties de ce notebook montrent des pixels Rubin DP2 : **ne jamais les versionner** (effacer les sorties avant de commiter).

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from res.config import Config
from res import viz

CONFIG = Path("../config/local_demo.yaml")
cfg = Config.load(CONFIG)
cc = cfg["cuts"]
table = pd.read_parquet(cfg.data("30_processed", "table.parquet"))
plt.rcParams["figure.dpi"] = 110
print(f"{len(table):,} objets dans le catalogue")
stamps, smeta = viz.load_stamps(cfg)
bits = smeta["plane_bits"]
T = table.set_index("object_id")

## Entonnoir

In [ ]:
f = viz.funnel(table)
display(f.style.format({"restants": "{:,}", "fraction retirée": "{:.1%}"}))
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(f["étape"][::-1], f["restants"][::-1], color="0.4")
ax.set_xscale("log"); ax.set_xlabel("objets restants")
plt.tight_layout()

À partir de « dans la région téléchargée », les pourcentages portent sur les cibles qui ont des pixels. Entonnoir restreint à ces cibles :

In [ ]:
f2 = viz.funnel(table, viz.STEPS[3:], start_mask=viz.stage_mask(table, "in_region"))
f2.style.format({"restants": "{:,}", "fraction retirée": "{:.1%}"})

## Grandeurs continues autour des seuils

Combien de cibles passeraient à un autre seuil, sans rien reconstruire.

In [ ]:
R = table[table["in_region"]]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
v = R["cut5_value"].replace(np.inf, 99)
k = np.linspace(0, 10, 200)
axes[0].plot(k, [(v > x).mean() for x in k], c="0.2"); axes[0].axvline(cc["k_ellipse"], c="C3", ls="--")
axes[0].set_xlabel("k (rayon réduit du pixel abîmé le plus proche)"); axes[0].set_ylabel("fraction qui passe")
axes[1].hist(np.log10(R["cut6_value"].clip(lower=0.1)), bins=50, color="0.4"); axes[1].axvline(np.log10(cc["snr_min"]), c="C3", ls="--")
axes[1].set_xlabel("log10 S/N(i)")
for ax, col, name in [(axes[2], "cut8_value", "|Δe| entre ouvertures 29 / 58 px"), (axes[3], "delta_e_mask", "|Δe| au masquage")]:
    x = R[col].dropna()
    t = np.logspace(-4, 0, 200)
    ax.semilogx(t, [(x <= s).mean() for s in t], c="0.2"); ax.axvline(cc["delta_e_max"], c="C3", ls="--")
    ax.set_xlabel(name); ax.set_title(f"évaluées : {len(x)} / {len(R)}", fontsize=8)
plt.tight_layout()

## Coupure 4 : pixels non exposés (en rouge)

In [ ]:
rej = R[~R["cut4_pass"].astype(bool)]
fig, axes = viz.grid(max(len(rej), 1))
for ax, row in zip(axes, rej.itertuples()):
    s = stamps[row.object_id]
    viz.show(ax, s["flux_raw"][3], f"{row.object_id}\nNO_DATA : {row.cut4_value} px")
    viz.overlay(ax, np.any(s["maskbits"] & bits["NO_DATA"], axis=0), "red")

## Coupure 5 : pixels abîmés (couleur par plan) et k × ellipse (tirets)

In [ ]:
colors = {"SATURATED": "red", "COSMIC_RAY": "orange", "INTERPOLATED": "cyan", "DETECTION_EDGE": "magenta"}
rej = R[~R["cut5_pass"].astype(bool)].sort_values("cut5_value").head(24)
fig, axes = viz.grid(len(rej))
for ax, row in zip(axes, rej.itertuples()):
    s = stamps[row.object_id]
    viz.show(ax, s["flux_raw"][3], f"r min = {row.cut5_value:.2f}")
    for p, col in colors.items():
        viz.overlay(ax, np.any(s["maskbits"] & bits[p], axis=0), col, 0.6)
    xc, yc = viz.stamp_center(row)
    viz.draw_ellipse(ax, xc, yc, row.ellipse_a, row.ellipse_b, row.ellipse_theta, 1, ec="C0", lw=0.8)
    viz.draw_ellipse(ax, xc, yc, row.ellipse_a, row.ellipse_b, row.ellipse_theta, cc["k_ellipse"], ec="C0", lw=0.8, ls="--")
print({p: c for p, c in colors.items()})

## Coupure 8 : la vignette de 59 px est-elle assez grande ?

On découpe autour de la cible une vignette de diagnostic **deux fois plus grande** (117 × 117 px), masquée de la même façon, et on mesure la forme de la cible dans deux ouvertures circulaires : 29 px de rayon (la vignette du jeu) et 58 px. Si les deux mesures diffèrent de plus de 0,02, de la lumière venue d'au-delà de la vignette déforme la cible.

**Légende** — titre : |Δe| entre les deux ouvertures ; **cercles rouges** : ouvertures de 29 et 58 px ; **carré pointillé** : la vignette de 59 px du jeu ; **ellipse bleue** : la cible ; **vert** : pixels remplacés par du ciel (compagnes masquées).

La mesure (moments adaptatifs de galsim) ne regarde que les pixels proches de la galaxie : pour la plupart des cibles, l'ouverture de 29 px contient déjà tout le poids et l'écart est nul. Les plus grands écarts, souvent des cibles très faibles :

In [ ]:
from matplotlib.patches import Rectangle
top = R[R["cut8_evaluated"].astype(bool)].sort_values("cut8_value", ascending=False).head(12)
fig, axes = viz.grid(len(top), ncols=6, size=2.4)
n, nd = smeta["stamp_px"], smeta["diag_px"]
for ax, row in zip(axes, top.itertuples()):
    s = stamps[row.object_id]
    if "diag_flux" not in s:
        continue
    viz.show(ax, s["diag_flux"], f"|Δe| = {row.cut8_value:.3f}")
    viz.overlay(ax, s["diag_need"], "C2", 0.25)
    xc, yc = viz.stamp_center(row, n=nd)
    for r_ in cc["aperture_radii"]:
        viz.draw_circle(ax, xc, yc, r_, ec="C3", lw=0.8)
    h = (nd - n) // 2
    ax.add_patch(Rectangle((h - 0.5, h - 0.5), n, n, fill=False, ec="0.2", ls=":", lw=0.8))
    viz.draw_ellipse(ax, xc, yc, row.ellipse_a, row.ellipse_b, row.ellipse_theta, ec="C0", lw=0.9)

## Biais : la mesure de forme ne doit pas trier les galaxies

Distributions après la coupure 2 (cibles de la région), puis dans le jeu fini. Distance de Kolmogorov–Smirnov sur |e| à comparer au plancher de bruit.

In [ ]:
from scipy.stats import ks_2samp
a_ = R
b_ = table[viz.stage_mask(table, "mask")]
size = lambda d: np.sqrt(d["ellipse_a"] * d["ellipse_b"]) * 0.2
emod = lambda d: np.hypot(d["e1_moments"], d["e2_moments"])
fig, axes = plt.subplots(1, 3, figsize=(14, 3.3))
for ax, f_, name, bins in [(axes[0], lambda d: d["i_cModel_mag"], "i [mag]", np.linspace(18, 28, 40)),
                           (axes[1], size, "taille de l'ellipse [arcsec]", np.linspace(0, 2, 40)),
                           (axes[2], emod, "|e| des moments", np.linspace(0, 1, 30))]:
    ax.hist(f_(a_), bins=bins, histtype="step", density=True, label=f"région ({len(a_)})", color="0.5")
    ax.hist(f_(b_), bins=bins, histtype="step", density=True, label=f"jeu fini ({len(b_)})", color="C3")
    ks = ks_2samp(f_(a_).dropna(), f_(b_).dropna())
    ax.set_xlabel(name); ax.set_title(f"KS = {ks.statistic:.3f}", fontsize=9)
axes[0].legend(fontsize=8)
plt.tight_layout()